# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Groq** para tener acceso a inferencia con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1EyWl33ZyAWuyMKXz_mr9aaNu3JF8e8vS?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [1]:
# Instalar cliente de Groq y leer API key desde Colab Secrets
!pip install groq --quiet

from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 1.9 MB/s eta 0:00:00
Cliente de Groq inicializado correctamente.


## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

In [2]:
# Prompt de clasificación en modo zero-shot
prompt_zero_shot = "Clasifica el sentimiento de esta reseña en Positivo, Negativo o Mixto: 'El envío llegó tarde pero el producto es excelente.' Respuesta muy breve y corta."

response_zero = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_zero_shot}]
)

print("Zero-shot:", response_zero.choices[0].message.content)

Zero-shot: Mixto.


In [3]:
# Prompt de clasificación en modo few-shot
prompt_few_shot = """Clasifica el sentimiento de cada reseña como Positivo, Negativo o Mixto.

Reseña: "Me encantó, llegó rápido y en perfecto estado."
Sentimiento: Positivo

Reseña: "Nunca llegó mi pedido, pésimo servicio."
Sentimiento: Negativo

Reseña: "El envío llegó tarde pero el producto es excelente."
Sentimiento:"""

response_few = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_few_shot}]
)

print("Few-shot:", response_few.choices[0].message.content)

Few-shot: Sentimiento: Mixto


### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

In [7]:
# Razonamiento paso a paso (chain-of-thought)
problema = (
    "Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "
    "ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tiempo tarda el segundo tren en "
    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)

response_cot = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": problema}]
)

print(response_cot.choices[0].message.content)

**Paso a paso:**

1. **Define el instante de referencia.**  
   - Sea \(t=0\) el momento en que sale el primer tren de la ciudad A.  
   - El segundo tren sale 2 h después, es decir, a \(t=2\) h.

2. **Expresa las distancias recorridas en función de \(t\).**  
   - Para el primer tren:  
     \[
     d_1 = 80\;(\text{km/h}) \times t
     \]
   - Para el segundo tren, solo empieza a moverse a partir de \(t=2\).  
     Si \(t\) es el tiempo transcurrido desde el **inicio** (tiempo absoluto), entonces su tiempo de marcha es \(t-2\).  
     \[
     d_2 = 120\;(\text{km/h}) \times (t-2)
     \]

3. **Condición de encuentro.**  
   El segundo tren alcanza al primero cuando sus distancias desde la ciudad A son iguales:
   \[
   d_1 = d_2 \quad\Longrightarrow\quad
   80\,t = 120\,(t-2)
   \]

4. **Resuelve la ecuación.**  
   \[
   80t = 120t - 240 \\
   120t - 80t = 240 \\
   40t = 240 \\
   t = \frac{240}{40} = 6 \text{ h}
   \]

5. **Interpreta el resultado.**  
   - El valor \(t=6\) h es e

### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

In [8]:
# Preguntar algo que el modelo no pudo haber visto en su entrenamiento y observar si alucina
prompt_desconocido = (
    "¿Cuál fue el resultado de la final del hackathon interno de DEV.F del 14 de agosto de "
    "2026? Respuesta muy breve y corta."
)

response_alucinacion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_desconocido}]
)

print(response_alucinacion.choices[0].message.content)

Lo siento, no dispongo de esa información.


## **RAG: BUSCAR ANTES DE RESPONDER**

RAG separa el proceso en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de una base de conocimiento propia (usando *embeddings* y similitud semántica); después, esos fragmentos se le entregan al modelo junto con la pregunta para generar la respuesta.

In [9]:
# Instalar sentence-transformers
!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np

In [10]:
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en su "
    "empaque original.",
    "Los envíos internacionales no tienen devolución gratuita; el cliente cubre el costo de "
    "envío de regreso.",
    "Los productos en oferta o liquidación no son elegibles para devolución, solo para "
    "cambio de talla."
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generados: (3, 384)


In [11]:
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese el más relevante
def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

pregunta = "¿Puedo devolver algo que compré en oferta?"
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)

Fragmento recuperado: Los productos en oferta o liquidación no son elegibles para devolución, solo para cambio de talla.


In [12]:
# Enviar la pregunta junto con el fragmento recuperado al modelo y mostrar la respuesta con RAG
prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda. Si la política no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_rag}]
)

print(response_rag.choices[0].message.content)

No, los productos en oferta o liquidación no son elegibles para devolución, solo se pueden cambiar de talla.


# **CHALLENGE: ASISTENTE DE POLÍTICAS CON RAG**

Una vez visto el ***Hands-On: Prompt Engineering y Sistemas RAG***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño asistente con RAG sobre un documento propio, comparando la respuesta **con RAG** contra la respuesta **sin RAG** para la misma pregunta. En esta solución se usa como base de conocimiento el propio reglamento de evaluación del curso IA Aplicada con Llama.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

## **INSTRUCCIONES:**

**1. Define tu base de conocimiento y genera embeddings:**

* Lee la API key previamente configurada desde **Colab Secrets** e instala/importa las librerías necesarias.

* Construye una lista llamada `documentos` con 3 fragmentos de un documento real de tu propio contexto (reglamento, políticas, FAQs, etc.) y genera sus embeddings con `sentence-transformers`.

In [30]:
# Leer API key, instalar e importar librerías
!pip install groq --quiet
!pip install sentence-transformers --quiet
!pip install pypdf

from groq import Groq
from google.colab import userdata
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import numpy as np

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print("Cliente de Groq inicializado correctamente.")

Cliente de Groq inicializado correctamente.


In [68]:
# Definir la lista documentos y generar sus embeddings
# El archivo de la ficha tecnica lo puede descargar de: https://www.mexia.gob.mx/wp-content/uploads/2026/07/Ficha-tecnica-Meta.pdf
# Una vez que halla descargado el archivo este debe ser accesible desde el notebook
archivo_pdf='/content/Ficha-tecnica-Meta.pdf'

try:
  reader = PdfReader(archivo_pdf)
  pdf_texts = [p.extract_text().strip() for p in reader.pages]

  # Filter the empty strings
  documentos  = [text for text in pdf_texts if text]

  modelo_challenge_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
  embeddings_docs = modelo_challenge_embeddings.encode(documentos)
  print("Embeddings generados:", embeddings_docs.shape)
except Exception as e:
  print(f"Error al leer el archivo pdf: {e}")
  print("Please download file before continue")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embeddings generados: (3, 384)


**2. Recupera el fragmento relevante:** Define una función `buscar_fragmento(pregunta)` que calcule la similitud coseno y regrese el fragmento más relevante para una pregunta dada.

In [69]:
# Definir la función buscar_fragmento
def busca_fragmento(pregunta):
    embedding_pregunta = modelo_challenge_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_docs, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

pregunta = "¿cual es el nivel educativo minimo para el Curso de Inteligencia Artificial Aplicada con Llama?"
fragmento_recuperado = busca_fragmento(pregunta)
print("Fragmento recuperado:", fragmento_recuperado)


Fragmento recuperado: ● Masterclass 3: Fine-tuning y Evaluación de Modelos (2 
hrs). 
● Masterclass 4: Pipeline Completo — De Datos a 
Modelo Desplegado (2 hrs). 
● Hackathon 1: Construir un Asistente Inteligente con 
Llama (4 hrs). 
● E-learning Módulo 1 (14 hrs): Setup de entorno, 
PyTorch, tokenización, embeddings, fine -tuning con 
LoRA, métricas de evaluación, integración end -to-end 
y preparación para hackathon. 
 
Módulo 2: Automatización Inteligente con WhatsApp Cloud 
API + Llama — 26 hrs 
● Masterclass 5: WhatsApp Cloud API — Arquitectura y 
Casos de Uso (2 hrs). 
● Masterclass 6: Diseño de Agentes Conversacionales 
con Llama (2 hrs). 
● Masterclass 7: Integración Llama + WhatsApp como 
Canal de Entrega (2 hrs). 
● Masterclass 8: Proyecto Integrad or — Agente 
Funcional en Producción (2 hrs). 
● Hackathon 2: Agente de WhatsApp que Resuelve un 
Problema Real (4 hrs). 
● E-learning Módulo 2 (14 hrs): Setup WhatsApp Cloud 
API, webhooks, flujos conversacionales, gestión de 
est

**3. Genera la respuesta sin RAG:** Envía una pregunta real sobre tu documento directamente al modelo (sin ningún fragmento de contexto) y guarda la respuesta en `respuesta_sin_rag`.

In [70]:
# Consultar la pregunta sin RAG y guardar el resultado en respuesta_sin_rag
respuestas = dict()

prompt_de_control = (
    f"""'{pregunta}'
    'Respuesta muy breve y corta.'"""
)

response_alucinacion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_de_control}]
)

respuestas['respuesta_sin_rag'] = response_alucinacion.choices[0].message.content

**4. Genera la respuesta con RAG:** Recupera el fragmento relevante con tu función y envía la pregunta junto con ese fragmento al modelo. Guarda la respuesta en `respuesta_con_rag`.

In [71]:
# Consultar la pregunta con RAG y guardar el resultado en respuesta_con_rag
prompt_rag = f"""Responde la pregunta del usuario usando SOLO la información de la Ficha técnica del curso. Si la información no cubre la pregunta, dilo claramente.

Política: {fragmento_recuperado}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_rag}]
)
respuestas['respuesta_con_rag'] = response_rag.choices[0].message.content

**5. Compara y concluye:** Imprime ambas respuestas y concluye cuál de las dos evitó mejor una alucinación o dio una respuesta más precisa.

In [72]:
# Mostrar ambas respuestas para comparar
print(f"A la pregunta: {pregunta}, se obtuvieron las siguientes respuestas:\n")
print(f"Sin RAG: {respuestas['respuesta_sin_rag']}\n")
print(f"Con RAG: {respuestas['respuesta_con_rag']}\n")

A la pregunta: ¿cual es el nivel educativo minimo para el Curso de Inteligencia Artificial Aplicada con Llama?, se obtuvieron las siguientes respuestas:

Sin RAG: Licenciatura (o su equivalente).

Con RAG: Preparatoria o Bachillerato terminado.

